# Task 5: Mental Health Support Chatbot (Fine-Tuned)

### Dataset

In [13]:
from huggingface_hub import notebook_login

notebook_login()

In [2]:
dataset_name = 'Estwld/empathetic_dialogues_llm'

from datasets import load_dataset
dataset = load_dataset(dataset_name)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.22M [00:00<?, ?B/s]

data/valid-00000-of-00001.parquet:   0%|          | 0.00/806k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/798k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19533 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/2770 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2547 [00:00<?, ? examples/s]

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['conv_id', 'situation', 'emotion', 'conversations'],
        num_rows: 19533
    })
    valid: Dataset({
        features: ['conv_id', 'situation', 'emotion', 'conversations'],
        num_rows: 2770
    })
    test: Dataset({
        features: ['conv_id', 'situation', 'emotion', 'conversations'],
        num_rows: 2547
    })
})

In [4]:
print(dataset['train'], '\n')

print(dataset['train'][0], '\n')

print(dataset['train'][0]['conversations'], '\n')

Dataset({
    features: ['conv_id', 'situation', 'emotion', 'conversations'],
    num_rows: 19533
}) 

{'conv_id': 'hit:0_conv:1', 'situation': 'I remember going to the fireworks with my best friend. There was a lot of people, but it only felt like us in the world.', 'emotion': 'sentimental', 'conversations': [{'content': 'I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people, we felt like the only people in the world.', 'role': 'user'}, {'content': 'Was this a friend you were in love with, or just a best friend?', 'role': 'assistant'}, {'content': 'This was a best friend. I miss her.', 'role': 'user'}, {'content': 'Where has she gone?', 'role': 'assistant'}, {'content': 'We no longer talk.', 'role': 'user'}, {'content': 'Oh was this something that happened because of an argument?', 'role': 'assistant'}]} 

[{'content': 'I remember going to see the fireworks with my best friend. It was the 

### Pretrained Model

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

pretrained_name = "distilbert/distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(pretrained_name)
model = AutoModelForCausalLM.from_pretrained(pretrained_name)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilbert/distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

### Special tokens

In [6]:
# since gpt2's tokenizer doesn't have a pad token
tokenizer.add_special_tokens({
    'pad_token': '[PAD]',
    'additional_special_tokens': ['[USER]', '[ASSISTANT]']
})

model.resize_token_embeddings(len(tokenizer))

print(tokenizer.truncation_side)
tokenizer.truncation_side = 'left'

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


right


### Preprocessing dataset
- formatting: converting conversations arrays for each example to single strings
- tokenizing

In [7]:
tokenizer.eos_token

'<|endoftext|>'

In [8]:
def format_example(example):
    text = ''

    # if example['situation']:
    #     text += f'Situation of user: {example["situation"]}\n'

    # if example['emotion']:
    #     text += f'User is feeling: {example["emotion"]}\n\n'

    for msg in example['conversations']:
        if msg['role'] == 'user':
            text += f'[USER] {msg["content"]} '
        else:
            # if msg['role'] == 'assistant':
            text += f'[ASSISTANT] {msg["content"]}{tokenizer.eos_token} '

    return {'text': text}

# .map in dataset applies it to each example in each of the subdatasets (train, test and val)
formatted_dataset = dataset.map(format_example, remove_columns=['conv_id', 'situation', 'emotion', 'conversations'])

Map:   0%|          | 0/19533 [00:00<?, ? examples/s]

Map:   0%|          | 0/2770 [00:00<?, ? examples/s]

Map:   0%|          | 0/2547 [00:00<?, ? examples/s]

In [9]:
formatted_dataset['train'][0]['text']

'[USER] I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people, we felt like the only people in the world. [ASSISTANT] Was this a friend you were in love with, or just a best friend?<|endoftext|> [USER] This was a best friend. I miss her. [ASSISTANT] Where has she gone?<|endoftext|> [USER] We no longer talk. [ASSISTANT] Oh was this something that happened because of an argument?<|endoftext|> '

In [10]:
assistant_token_id = tokenizer.convert_tokens_to_ids("[ASSISTANT]")

def tokenize_example(examples):
    tokenized = tokenizer(
        examples['text'],
        truncation=True, # trim instead of raising error if longer than max_length
        max_length=512, # distilgpt2 supports upto 1024 smaller like 256 would be faster to train, but shorter context
        return_tensors=None,
        padding='max_length'
    )

    # tokenized looks like this since it is batched
    # {
    #     'input_ids': [
    #         [101, 2023, 2003, 102], # example 1
    #         [101, 3452, 2100, 102], # example 2
    #         ...
    #     ],
    #     'attention_mask': [
    #         [1, 1, 1, 1],           # example 1
    #         [1, 1, 1, 1],           # example 2
    #         ...
    #     ]
    # }

    # masking all tokens except assistant responses so that it learns only what assistant should say, not copying user prompts
    batch_labels = []

    for input_ids in tokenized['input_ids']: # i.e. for each batch
        assistant_start_found = False
        labels = list(input_ids) # copy
        for i, id in enumerate(labels):
            if not assistant_start_found:
                if id == assistant_token_id:
                    assistant_start_found = True
                labels[i] = -100
            else:
                if id == tokenizer.eos_token_id:
                    assistant_start_found = False

        # if len(labels) != len(input_ids):
        #     raise Exception("Inconsistent lengths")

        batch_labels.append(labels)

    tokenized['labels'] = batch_labels
    # shifting to be handled inside collator
    return tokenized

# "input_ids" is what the model is given
# "labels" is what the model should output
# both should be same except that labels should be shifted by 1 (which is done internally automatically)

tokenized_dataset = formatted_dataset.map(tokenize_example, batched=True, remove_columns=['text'])
# could be combined within formatting function, but didn't do it for readability and single responsibility principle
# batched=True sends { "text": [..., ..., ...] } instead of a single value of "text"


Map:   0%|          | 0/19533 [00:00<?, ? examples/s]

Map:   0%|          | 0/2770 [00:00<?, ? examples/s]

Map:   0%|          | 0/2547 [00:00<?, ? examples/s]

In [11]:
for i in range(5):
    for key in tokenized_dataset['train'][i]:
        print(key, len(tokenized_dataset['train'][i][key]))
    print()

input_ids 512
attention_mask 512
labels 512

input_ids 512
attention_mask 512
labels 512

input_ids 512
attention_mask 512
labels 512

input_ids 512
attention_mask 512
labels 512

input_ids 512
attention_mask 512
labels 512



### Trainer & Arguments

In [14]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

hub_model_id = "mafgit/empathetic-distilgpt2"

train_args = TrainingArguments(
    output_dir='./empathetic-distilgpt2',
    num_train_epochs=3, # more can cause overfitting and repitition
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    fp16=True,
    weight_decay=0.01,
    learning_rate=3e-5,
    optim='adamw_torch',
    logging_steps=100,
    save_strategy='epoch',
    save_total_limit=2,
    report_to='none',
    push_to_hub=True,
    hub_model_id=hub_model_id,
    hub_strategy="every_save",
)

# # when trainer asks for a batch, data collator takes a batch from dataset and pads them to set lengths and hands it to GPU
# data_collator = DataCollatorForLanguageModeling(
#     tokenizer=tokenizer,
#     mlm=False,
#     pad_to_multiple_of=8 # efficient/faster on GPU with fp16=True
# )

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['valid'],
    # data_collator=data_collator, # handles batching and padding
    processing_class=tokenizer # so that it saves this in huggingface along with model as well
)

In [15]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50257}.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,3.380479
200,3.221229
300,3.130574
400,3.101508
500,3.108553
600,3.093034
700,3.042545
800,3.061210
900,3.049995
1000,3.019570


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3663, training_loss=2.954312198773258, metrics={'train_runtime': 2234.5364, 'train_samples_per_second': 26.224, 'train_steps_per_second': 1.639, 'total_flos': 7655864143970304.0, 'train_loss': 2.954312198773258, 'epoch': 3.0})

In [16]:
test_results = trainer.evaluate(eval_dataset=tokenized_dataset['test'])
test_results

{'eval_loss': 2.8986825942993164,
 'eval_runtime': 34.8718,
 'eval_samples_per_second': 73.039,
 'eval_steps_per_second': 9.148,
 'epoch': 3.0}

### Saving/pushing model & tokenizer

In [17]:
# trainer.save_model('./empathetic-distilgpt2-final')
trainer.push_to_hub(
    model_name='empathetic-distilgpt2',
    language='en',
    finetuned_from='distilbert/distilgpt2',
    tasks='text-generation',
    dataset=dataset_name,
    tags=["empathy", "dialogue", "conversational-ai"]
)

# tokenizer.save_pretrained('./empathetic-distilgpt2-final')
tokenizer.push_to_hub('empathetic-distilgpt2') # in case processing_class isn't set in trainer

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tilgpt2/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...tilgpt2/model.safetensors:  46%|####6     |  152MB /  328MB            

README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/mafgit/empathetic-distilgpt2/commit/fd03a261e2410bac920a38c49375231d0e8d3bfc', commit_message='Upload tokenizer', commit_description='', oid='fd03a261e2410bac920a38c49375231d0e8d3bfc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/mafgit/empathetic-distilgpt2', endpoint='https://huggingface.co', repo_type='model', repo_id='mafgit/empathetic-distilgpt2'), pr_revision=None, pr_num=None)